# 01 — Exploratory data analysis
Goal: understand sizes, formats, noise and match structure **before** tuning anything.
Everything here is read-only; pipeline logic lives in `src/`.

In [ ]:
# --- setup: run from the repo root or from notebooks/ ---
import sys, os, json, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
# pick the first data dir that exists; full data is ~12M rows/split, so EDA defaults to samples
for cand in ["dataset_sample", "dataset_region", "dataset", "fake_dataset"]:
    if (ROOT / cand / "train").exists():
        DATA = ROOT / cand
        break
CACHE = ROOT / "cache"                  # pipeline cache (blocking .npz, oof_train.pkl)
CACHE = ROOT / "cache"
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)
print("data:", DATA)

In [ ]:
from data import read_tsv
raw = {}
for split in ["train", "test"]:
    for s in [1, 2, 3]:
        raw[(split, s)] = read_tsv(DATA / split / f"{split}_source{s}.tsv")
gt = read_tsv(DATA / "train" / "train_ground_truth.tsv")
pd.DataFrame({f"{sp} S{s}": [len(df)] for (sp, s), df in raw.items()}, index=["rows"])

## Schema and data quality

In [ ]:
rows = []
for (sp, s), df in raw.items():
    rows.append(dict(split=sp, source=s, rows=len(df),
                     dup_ids=df["entity_id"].duplicated().sum(),
                     bad_prefix=(~df["entity_id"].str.startswith(f"S{s}-")).sum(),
                     empty_name=(df["business_name"].str.strip() == "").mean(),
                     empty_addr=(df["business_address"].str.strip() == "").mean(),
                     empty_country=(df["country"].str.strip() == "").mean()))
pd.DataFrame(rows)

## Countries — which labels exist, and which appear only in test

In [ ]:
cc = pd.concat([df["country"].value_counts().rename(f"{sp} S{s}") for (sp, s), df in raw.items()], axis=1).fillna(0).astype(int)
display(cc)
train_c = set().union(*[set(raw[("train", s)]["country"]) for s in [1, 2, 3]])
test_c = set().union(*[set(raw[("test", s)]["country"]) for s in [1, 2, 3]])
print("countries only in test:", test_c - train_c)
print("raw label variants (check for 'USA' vs 'US' etc.):", sorted(train_c | test_c))

## Match structure (train ground truth)

In [ ]:
gt["ids"] = gt["matched_entity_ids"].map(lambda s: [x for x in s.split(",") if x])
gt["n"] = gt["ids"].map(len)
gt["n_s2"] = gt["ids"].map(lambda l: sum(x.startswith("S2-") for x in l))
gt["n_s3"] = gt["ids"].map(lambda l: sum(x.startswith("S3-") for x in l))
print(f"S1 entities: {len(gt)}   singletons: {(gt.n == 0).mean():.1%}")
display(gt["n"].value_counts().sort_index().to_frame("S1 entities by #matches").T)
display(pd.crosstab(gt.n_s2, gt.n_s3).rename_axis(index="#S2", columns="#S3"))

In [ ]:
# Is it one-to-one? (each S2/S3 record linked to at most one S1)  -> decides whether the one_to_one decoder is valid
from collections import Counter
cnt = Counter(x for l in gt["ids"] for x in l)
multi = {k: v for k, v in cnt.items() if v > 1}
print("S2/S3 ids linked to >1 S1:", len(multi))
for split_s in [2, 3]:
    ids = raw[("train", split_s)]["entity_id"]
    print(f"S{split_s}: {ids.isin(cnt).mean():.1%} of records are linked to some S1 (rest are distractors)")

## Look at real matched pairs side by side

In [ ]:
s1 = raw[("train", 1)].set_index("entity_id")
s23 = pd.concat([raw[("train", 2)], raw[("train", 3)]]).set_index("entity_id")
country = s1["country"]

def show_entity(sid):
    """Print an S1 record and all its true matches."""
    r = s1.loc[sid]
    print(f"{sid} [{r.country}]  {r.business_name}  ||  {r.business_address}")
    for m in gt.set_index("source1_entity_id").loc[sid, "ids"]:
        q = s23.loc[m]
        print(f"   {m} [{q.country}]  {q.business_name}  ||  {q.business_address}")

for sid in gt[gt.n > 0].sample(12, random_state=1)["source1_entity_id"]:
    show_entity(sid); print()

## Names: lengths, legal suffixes, frequent tokens by country

In [ ]:
from normalize import normalize_name, core_name, LEGAL_SUFFIXES
allnames = pd.concat([raw[("train", s)].assign(src=s) for s in [1, 2, 3]])
allnames["name_n"] = allnames["business_name"].map(normalize_name)
allnames["ntok"] = allnames["name_n"].str.split().map(len)
display(allnames.groupby(["country", "src"])["ntok"].describe()[["mean", "50%", "max"]])
tok = allnames["name_n"].str.split().explode()
print("most common legal-suffix tokens:", tok[tok.isin(LEGAL_SUFFIXES)].value_counts().head(20).to_dict())
suffix_like = tok[~tok.isin(LEGAL_SUFFIXES)].value_counts().head(40)
print("most common other tokens (candidates for the stop/suffix list):", suffix_like.to_dict())

## Addresses: postcodes, numbers, formats by country

In [ ]:
from normalize import normalize_address, first_number
for (sp, s), df in raw.items():
    fn = df["business_address"].map(lambda a: first_number(normalize_address(a)))
    print(f"{sp} S{s}: has a number:", (fn != "").groupby(df["country"]).mean().round(3).to_dict(),
          " empty address:", (df["business_address"].str.strip() == "").groupby(df["country"]).mean().round(3).to_dict())

In [ ]:
# raw vs normalised addresses, a few per country and source (look for formats we don't handle yet)
for c in sorted(test_c):
    print(f"==== {c}")
    for s in [1, 2, 3]:
        df = raw[("test", s)]
        smp = df[df.country == c].sample(min(3, (df.country == c).sum()), random_state=0)
        for a in smp["business_address"]:
            print(f"  S{s} {a!r:70.70}  ->  {normalize_address(a)!r}")

## How similar are true pairs vs random pairs? (sanity check on normalisation)

In [ ]:
from sims import jaro_winkler
pairs = [(sid, m) for sid, l in zip(gt.source1_entity_id, gt.ids) for m in l]
rng = np.random.default_rng(0)
pairs = [pairs[i] for i in rng.choice(len(pairs), min(3000, len(pairs)), replace=False)]
true_sim = [jaro_winkler(core_name(normalize_name(s1.loc[a, "business_name"])), core_name(normalize_name(s23.loc[b, "business_name"]))) for a, b in pairs]
rnd_b = s23.index[rng.choice(len(s23), len(pairs))]
rand_sim = [jaro_winkler(core_name(normalize_name(s1.loc[a, "business_name"])), core_name(normalize_name(s23.loc[b, "business_name"]))) for (a, _), b in zip(pairs, rnd_b)]
plt.hist(true_sim, bins=40, alpha=.6, label="true pairs"); plt.hist(rand_sim, bins=40, alpha=.6, label="random pairs")
plt.xlabel("Jaro-Winkler on core name"); plt.legend(); plt.show()
print("true pairs with JW < 0.7 (names alone won't find them):", np.mean(np.array(true_sim) < 0.7).round(3))